In [1]:
import os
import tqdm
import librosa
import parselmouth
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
from parselmouth.praat import call
from src.display import display_audio_table

## Problem 1(a): Apply PSOLA

In [2]:
class AudioModifier:
    def __init__(self, audio_path, time_step=0.01, f0_min=75, f0_max=500):
        self.input_name = os.path.basename(audio_path).split(".")[0]
        self.snd = parselmouth.Sound(audio_path)
        self.time_step = time_step
        self.f0_min = f0_min
        self.f0_max = f0_max

    def create_manipulation(self):
        return call(self.snd, "To Manipulation", self.time_step, self.f0_min, self.f0_max)

    def apply_effect(self, effect):
        effect_type, factor = effect
        if effect_type == "pitch":
            return self.modify_pitch(factor)
        elif effect_type == "duration":
            return self.modify_duration(factor)
        elif effect_type == "monotone":
            return self.modify_monotone()
        else:
            raise ValueError(f"Unknown effect type: {effect_type}")
        
    def modify_pitch(self, ratio):
        # ratio = pitch after / pitch before
        m = self.create_manipulation()
        pt = call(m, "Extract pitch tier")
        call(pt, "Multiply frequencies", self.snd.xmin, self.snd.xmax, ratio)
        call([pt, m], "Replace pitch tier")
        new = call(m, "Get resynthesis (overlap-add)")
        return new

    def modify_duration(self, ratio):
        # ratio = duration after / duration before
        m = self.create_manipulation()
        dt = call(m, "Extract duration tier")
        call(dt, "Add point", self.snd.xmin + 0.001, ratio)
        call([dt, m], "Replace duration tier")
        new = call(m, "Get resynthesis (overlap-add)")
        return new

    def modify_monotone(self):
        m = self.create_manipulation()
        pitch = self.snd.to_pitch()
        f0_values = pitch.selected_array["frequency"]
        voiced_f0_values = f0_values[f0_values > 0]
        median_f0 = np.median(voiced_f0_values)

        flat = call("Create PitchTier", "flat", self.snd.xmin, self.snd.xmax)
        call(flat, "Add point", self.snd.xmin, median_f0)
        call([flat, m], "Replace pitch tier")
        new = call(m, "Get resynthesis (overlap-add)")
        return new

In [13]:
effects = {
    "Pitch ×1.5": ("pitch", 1.5),
    "Pitch ×0.75": ("pitch", 0.75),
    "Duration ×0.8": ("duration", 0.8),
    "Duration ×1.5": ("duration", 1.5),
    "Monotone": ("monotone", None)
}

In [14]:
output_dir = "./output/problem1/psola"
os.makedirs(output_dir, exist_ok=True)
samples = {
    './hw2_data/speech_clips/clean_lj.wav': [],
    './hw2_data/speech_clips/clean_jfk.wav': []
}
for sample in samples:
    audio_modifier = AudioModifier(sample)
    for effect_name, effect in effects.items():
        if effect[1] is None:
            out_name = os.path.join(output_dir, f"{audio_modifier.input_name}_{effect[0]}.wav")
        else:
            out_name = os.path.join(output_dir, f"{audio_modifier.input_name}_{effect[0]}_{effect[1]}.wav")
        new_audio = audio_modifier.apply_effect(effect)
        new_audio.save(out_name, "WAV")

        samples[sample].append((effect, out_name))

In [15]:
display_audio_table(list(effects.values()), ['Original'] + list(effects.keys()), samples)

Original,Pitch ×1.5,Pitch ×0.75,Duration ×0.8,Duration ×1.5,Monotone
clean_lj,,,,,
clean_jfk,,,,,


### Analysis

## Problem 1(b): PSOLA vs. resampling

## Problem 1(c): Your own voice